# What each column records

Every training row is one passenger. The test rows use the same clues and omit `Transported`.

| Column | What the cell means |
|---|---|
| `PassengerId` | `gggg_pp`: the group, then the person inside that group |
| `HomePlanet` | `Earth`, `Europa`, or `Mars` |
| `CryoSleep` | `True` if the passenger was in suspended animation and confined to the cabin |
| `Cabin` | `deck/number/side`. Side `P` is port. Side `S` is starboard |
| `Destination` | `TRAPPIST-1e`, `PSO J318.5-22`, or `55 Cancri e` |
| `Age` | Age in years |
| `VIP` | `True` if the passenger paid for VIP service |
| `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck` | The amount billed at that amenity |
| `Name` | The passenger's name, two words on every filled row |
| `Transported` | The fate. Present in `train.csv` only |

A blank cell is an empty string. It is a missing clue, not a planet, a deck, or a zero bill.

The first training row is passenger `0001_01`, Maham Ofracculy. Read it by hand before trusting a summary.

$$
\begin{aligned}
\text{PassengerId } 0001\_01 &\Rightarrow \text{group } 0001,\ \text{person } 01 \\
\text{Cabin } B/0/P &\Rightarrow \text{deck } B,\ \text{cabin number } 0,\ \text{side } P
\end{aligned}
$$

She is from Europa, she was awake, she was 39, she was not VIP, every bill is 0, and she was not transported.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Split the first id and the first cabin into the fields the string actually stores.
maham = train[0]
group, person = maham["PassengerId"].split("_")
deck, number, side = maham["Cabin"].split("/")
print(group, person, deck, number, side)
print(maham["HomePlanet"], maham["CryoSleep"], maham["Age"], maham["Transported"])
assert maham["PassengerId"] == "0001_01"
assert maham["Name"] == "Maham Ofracculy"
assert (group, person) == ("0001", "01")
assert (deck, number, side) == ("B", "0", "P")
assert maham["HomePlanet"] == "Europa"
assert maham["CryoSleep"] == "False"
assert maham["Age"] == "39.0"
assert maham["VIP"] == "False"
assert maham["Transported"] == "False"
for column in ("RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"):
    assert maham[column] == "0.0"


0001 01 B 0 P
Europa False 39.0 False


## Pitfall

The cabin number and the bill amounts are easy to treat as one numeric column. The cabin number is an index along a deck. A bill of `0.0` is a real zero. An empty bill cell is a different fact, and later rows contain empty bill cells.

## Summary

Thirteen clue columns, then one fate column that only the training file has. `0001_01` is group `0001`, person `01`, deck `B`, side `P`, and she stayed.
